<a href="https://colab.research.google.com/github/prasanna-venkatesh-m/tiny-transformer/blob/main/Tiny_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install the Gensim and NLTK

In [109]:
pip install gensim nltk

Import and download all the Packages

In [110]:
import gensim
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.tokenize import word_tokenize
import tensorflow as tf
import numpy as np

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


# ***ENCODER BLOCK***

Converting the sentance into token (Like spliting the words)

In [111]:
sentances = "The cat sat on the mat"
# converting the sentance into tokens (by words or subwords)
tokens = word_tokenize(sentances.lower(), language='english', preserve_line=True)
print(tokens)

# converting the tokens into sets
vocab = {}
i =0
for word in set(tokens):
  vocab[word]=i
  i = i+1
print(vocab)

vocab_ids = []
for word in tokens:
  vocab_ids.append(vocab[word])
print(vocab_ids)

['the', 'cat', 'sat', 'on', 'the', 'mat']
{'the': 0, 'on': 1, 'mat': 2, 'cat': 3, 'sat': 4}
[0, 3, 4, 1, 0, 2]


Convert it to Tensors

In [112]:
# converting the normal array to Tensor
ids_tensor = tf.constant(vocab_ids)

# vector embedding (for embedding vector layer)
vocab_size = len(vocab)
seq_len = len(vocab_ids)
embedding_dim = 64
embedding_layer = tf.keras.layers.Embedding(input_dim = vocab_size, output_dim = embedding_dim)

# convert the tokens to vector embeddings
token_embeddings = embedding_layer(ids_tensor)
print(token_embeddings.shape)

(6, 64)


# Positional Embeddings

In [113]:
# Initialize the pos_encoding with zeros (token_size, embedding_dimension)
pos_encoding = np.zeros((seq_len, embedding_dim))

# fill the pos_encoding
for i in range(seq_len):
  for j in range(embedding_dim):
    if i % 2 == 0:
      pos_encoding[i,j] = np.sin(i/(10000 ** (j / embedding_dim)))
    else:
      pos_encoding[i,j] = np.cos(i/ (10000 ** ((j-1)/embedding_dim)))

#convert to tensor
pos_encoding = tf.cast(pos_encoding, dtype=tf.float32)

# print pos encoding
print(pos_encoding.shape)

(6, 64)


# Adding the Token Embeddings and Positional Encoding to get Final Input

In [114]:
final_input = tf.add(token_embeddings , pos_encoding)
print(final_input.shape)

(6, 64)


# Self Attention

In [115]:
# we can initalize the dimension we want to set to Weights of Q,K,V
d = 64

# assign random weights to the Q,K,V
wQ = tf.Variable(tf.random.normal([embedding_dim, d]))
wK = tf.Variable(tf.random.normal([embedding_dim, d]))
wV = tf.Variable(tf.random.normal([embedding_dim, d]))

# multiply the final input with Weights to get Q,K,V
Q = tf.matmul(final_input, wQ)
K = tf.matmul(final_input, wK)
V = tf.matmul(final_input, wV)

print(Q.shape)
print(K.shape)
print(V.shape)

(6, 64)
(6, 64)
(6, 64)


**Scaled Dot-Product Attention**

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V
$$

Where:  
- \(Q\) = Query matrix  
- \(K\) = Key matrix  
- \(V\) = Value matrix  
- \(d_k\) = dimension of the keys  
- softmax = applied row-wise to get attention weights


In [116]:
# calculate the attention_score by multiplying Q.K^T
attention_scores = tf.matmul(Q, tf.transpose(K))

# divide the attention score by squareroot of dk
attention_scores = attention_scores / tf.sqrt(tf.cast(d, dtype=tf.float32))
print(f"Attention score shape : {attention_scores.shape}")

# take the softmax of the attentions scores
attention_weights = tf.nn.softmax(attention_scores, axis=-1)
print(f"Attention weights shape : {attention_weights.shape}")

# multiply with V to get the attention output
attention_output = tf.matmul(attention_weights, V)
print(f"Attention output shape : {attention_output.shape}")

Attention score shape : (6, 6)
Attention weights shape : (6, 6)
Attention output shape : (6, 64)


In [117]:
# layer normalization
layer_norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

# residual connection
attention_norm_output = layer_norm1(attention_output + final_input)

print(f"Transformer Attention output shape : {attention_norm_output.shape}")

Transformer Attention output shape : (6, 64)


Feed Forward Netword

In [118]:
#ffn

ffn_dim = 128

ffn = tf.keras.Sequential([
    tf.keras.layers.Dense(ffn_dim, activation='relu'),
    tf.keras.layers.Dense(embedding_dim)
])

ffn_output = ffn(attention_norm_output)
print(f"{ffn_output.shape}")

(6, 64)


In [119]:
#layer normalization + residual connection
layer_norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
encoder_output = layer_norm2(ffn_output + attention_norm_output)
print(f"Encoder output shape : {encoder_output.shape}")

Encoder output shape : (6, 64)


# ***DECODER BLOCK***

In [120]:
decoder_input = encoder_output

In [121]:
seq_len_2 = decoder_input.shape[0]

#create a masked array for decoder output
look_ahead_mask = []

for i in range(seq_len_2):
  row=[]
  for j in range(seq_len_2):
    if(i<j):
      row.append(1)
    else:
      row.append(0)
  look_ahead_mask.append(row)

look_ahead_mask = tf.constant(look_ahead_mask, dtype = tf.float32)

In [122]:
# assign random weights to the Q,K,V of decoder weights d1
wQ_d1 = tf.Variable(tf.random.normal([embedding_dim, d]))
wK_d1 = tf.Variable(tf.random.normal([embedding_dim, d]))
wV_d1 = tf.Variable(tf.random.normal([embedding_dim, d]))

# multiply the decoder input with Weights to get Q,K,V of d1
Q_d1 = tf.matmul(decoder_input, wQ_d1)
K_d1 = tf.matmul(decoder_input, wK_d1)
V_d1 = tf.matmul(decoder_input, wV_d1)

print(Q_d1.shape)
print(K_d1.shape)
print(V_d1.shape)

(6, 64)
(6, 64)
(6, 64)


In [123]:
# calculate the attention_score by multiplying Q_d1.K_d1^T
attention_scores_d1 = tf.matmul(Q_d1, tf.transpose(K_d1))

# divide the attention score by squareroot of dk_d1
attention_scores_d1 = attention_scores_d1 / tf.sqrt(tf.cast(d, dtype=tf.float32))
print(f"Attention score d1 shape : {attention_scores_d1.shape}")

#before taking the softmax add the look_ahead_mask
attention_scores_d1 = attention_scores_d1 + (look_ahead_mask * -1e9)

# take the softmax of the attentions scores
attention_weights_d1 = tf.nn.softmax(attention_scores_d1, axis=-1)
print(f"Attention weights shape : {attention_weights_d1.shape}")

# multiply with V to get the attention output
masked_attention_output = tf.matmul(attention_weights_d1, V_d1)
print(f"Masked self Attention output shape : {masked_attention_output.shape}")

Attention score d1 shape : (6, 6)
Attention weights shape : (6, 6)
Masked self Attention output shape : (6, 64)


In [124]:
# layer normalization
masked_layer_norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

# residual connection
masked_attention_norm_output = masked_layer_norm1(masked_attention_output + decoder_input)

print(f"Transformer Masked Attention output shape : {masked_attention_norm_output.shape}")

decoder_output = masked_attention_norm_output

Transformer Masked Attention output shape : (6, 64)


# ***CROSS ATTENTION***

In [125]:
# assign random weights to the Q,K,V of cross attention weights d2
wQ_d2 = tf.Variable(tf.random.normal([embedding_dim, d]))
wK_d2 = tf.Variable(tf.random.normal([embedding_dim, d]))
wV_d2 = tf.Variable(tf.random.normal([embedding_dim, d]))

# multiply the decoder input with Weights to get Q,K,V of d1
Q_d2 = tf.matmul(decoder_output, wQ_d2)
K_d2 = tf.matmul(encoder_output, wK_d2)
V_d2 = tf.matmul(encoder_output, wV_d2)

print(Q_d2.shape)
print(K_d2.shape)
print(V_d2.shape)

(6, 64)
(6, 64)
(6, 64)


In [130]:
# calculate the attention_score by multiplying Q_d1.K_d1^T
attention_scores_d2 = tf.matmul(Q_d2, tf.transpose(K_d2))

# divide the attention score by squareroot of dk_d2
attention_scores_d2 = attention_scores_d2 / tf.sqrt(tf.cast(d, dtype=tf.float32))
print(f"Attention score d1 shape : {attention_scores_d2.shape}")

# take the softmax of the attentions scores
attention_weights_d2 = tf.nn.softmax(attention_scores_d2, axis=-1)
print(f"Attention weights shape : {attention_weights_d2.shape}")

# multiply with V to get the attention output
cross_attention_output = tf.matmul(attention_weights_d2, V_d2)
print(f"Cross Attention output shape : {cross_attention_output.shape}")

Attention score d1 shape : (6, 6)
Attention weights shape : (6, 6)
Cross Attention output shape : (6, 64)


In [127]:
# layer normalization
cross_attn_layer_norm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

# residual connection
cross_attention_norm_output = cross_attn_layer_norm1(cross_attention_output + decoder_output)

print(f"Transformer Masked Attention output shape : {cross_attention_norm_output.shape}")

Transformer Masked Attention output shape : (6, 64)


In [128]:
#decoder_ffn

decoder_ffn_dim = 128

decoder_ffn = tf.keras.Sequential([
    tf.keras.layers.Dense(decoder_ffn_dim, activation='relu'),
    tf.keras.layers.Dense(embedding_dim)
])

decoder_ffn_output = decoder_ffn(cross_attention_norm_output)
print(f"{decoder_ffn_output.shape}")

(6, 64)


In [129]:
decoder_layer_norm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
decoder_output = decoder_layer_norm2(decoder_ffn_output + cross_attention_norm_output)
print(f"Transformer output shape : {decoder_output.shape}")

Transformer output shape : (6, 64)
